In [ ]:
# =====================================================================
# BƯỚC 1: IMPORT THƯ VIỆN
# =====================================================================
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import warnings
# Tắt cảnh báo để console gọn
warnings.filterwarnings('ignore')

print("BẮT ĐẦU KHỞI CHẠY HỆ THỐNG ĐÁNH GIÁ (NHÓM A, B, D)")

# =====================================================================
# BƯỚC 2: ĐỌC VÀ XỬ LÍ DỮ LIỆU
# =====================================================================
# Đọc file CSV
df = pd.read_csv('/content/Tong_hop_Qty_theo_F_SKU_Kem_PhanKhuc.csv')

# Xử lý khoảng trắng ở tên cột
df.columns = df.columns.str.strip()
TEN_COT_PHAN_KHUC = 'Phân khúc' if 'Phân khúc' in df.columns else 'Nhom_Phan_Khuc'

# Chuyển đổi định dạng thời gian
df['year_month'] = pd.to_datetime(df['year_month'])

# Chỉ lọc lấy nhóm A, B, D
target_segments = ['A — Core ML', 'B — Mid', 'D — Dormant']
df_filtered = df[df[TEN_COT_PHAN_KHUC].isin(target_segments)]

# =====================================================================
# BƯỚC 3: XÂY DỰNG CÁC MÔ HÌNH (CÁC HÀM HỖ TRỢ)
# =====================================================================
def prepare_ml_data(df_input):
    """Tạo biến trễ (Lag) và lịch (Month/Quarter) cho ML (Nhóm A & B)"""
    df_ml = df_input.copy()
    df_ml['Lag_1'] = df_ml['qty'].shift(1)
    df_ml['Lag_2'] = df_ml['qty'].shift(2)
    df_ml['Lag_3'] = df_ml['qty'].shift(3)
    df_ml['Month'] = df_ml['year_month'].dt.month
    df_ml['Quarter'] = df_ml['year_month'].dt.quarter
    return df_ml.dropna()

def ma_trend_forecast(ts, steps):
    """Moving Average kết hợp xu hướng (Nhóm B)"""
    ts = list(ts)
    if len(ts) < 3: return [np.mean(ts)] * steps
    ma3 = np.mean(ts[-3:])
    trend = (ts[-1] - ts[-3]) / 2 # Tính độ dốc
    return [max(0, ma3 + (trend * i)) for i in range(1, steps + 1)]

def croston_forecast(ts, steps, alpha=0.2):
    """Thuật toán Croston chuyên trị hàng đứt quãng (Nhóm D)"""
    ts = list(ts)
    z, idx = [], []
    for i, val in enumerate(ts):
        if val > 0:
            z.append(val)
            idx.append(i)
    if len(z) == 0: return [0.0] * steps

    p = [idx[0] + 1]
    for i in range(1, len(idx)): p.append(idx[i] - idx[i-1])

    z_hat, p_hat = z[0], p[0]
    for i in range(1, len(z)):
        z_hat = alpha * z[i] + (1 - alpha) * z_hat
        p_hat = alpha * p[i] + (1 - alpha) * p_hat

    forecast_val = z_hat / p_hat if p_hat > 0 else 0
    return [forecast_val] * steps


# =====================================================================
# BƯỚC 4 & 5: THI ĐẤU MÔ HÌNH, TÍNH TOÁN VÀ GHI NHẬN MÔ HÌNH TỐI ƯU
# =====================================================================
results = []
segments = df_filtered[TEN_COT_PHAN_KHUC].dropna().unique()

for segment in segments:
    fskus = df_filtered[df_filtered[TEN_COT_PHAN_KHUC] == segment]['F_SKU'].unique()
    print(f"\n Đang thi đấu tại Phân khúc: [{segment}] ({len(fskus)} FSKU)")

    for fsku in fskus:
        df_sku = df_filtered[df_filtered['F_SKU'] == fsku].sort_values('year_month').reset_index(drop=True)

        # Bỏ qua nếu dữ liệu dưới 6 tháng (Không đủ để tạo tập Train và Test 3 tháng)
        if len(df_sku) < 6:
            continue

        train_df = df_sku.iloc[:-3].copy()
        test_df = df_sku.iloc[-3:].copy()
        y_true = test_df['qty'].values

        mapes, rmses = {}, {}

        def evaluate(name, preds):
            # Tính RMSE và MAPE
            rmses[name] = np.sqrt(mean_squared_error(y_true, preds))
            mapes[name] = mean_absolute_percentage_error(y_true, preds) * 100

        # --------- LUẬT THI ĐẤU CHO TỪNG NHÓM ---------
        if segment == 'A — Core ML':
            df_train_clean = prepare_ml_data(train_df)
            if len(df_train_clean) > 0:
                X_train, y_train = df_train_clean[['Lag_1', 'Lag_2', 'Lag_3', 'Month', 'Quarter']], df_train_clean['qty']

                m_xgb = xgb.XGBRegressor(n_estimators=50, learning_rate=0.1, random_state=42).fit(X_train, y_train)
                m_lgb = lgb.LGBMRegressor(n_estimators=50, learning_rate=0.1, random_state=42, verbose=-1).fit(X_train, y_train)

                xgb_p, lgb_p = [], []
                last_xgb, last_lgb = train_df['qty'].tolist(), train_df['qty'].tolist()

                for i in range(3):
                    m, q = test_df['year_month'].iloc[i].month, test_df['year_month'].iloc[i].quarter
                    p_x = m_xgb.predict(pd.DataFrame([[last_xgb[-1], last_xgb[-2], last_xgb[-3], m, q]], columns=['Lag_1', 'Lag_2', 'Lag_3', 'Month', 'Quarter']))[0]
                    p_l = m_lgb.predict(pd.DataFrame([[last_lgb[-1], last_lgb[-2], last_lgb[-3], m, q]], columns=['Lag_1', 'Lag_2', 'Lag_3', 'Month', 'Quarter']))[0]
                    xgb_p.append(max(0, p_x)); last_xgb.append(max(0, p_x))
                    lgb_p.append(max(0, p_l)); last_lgb.append(max(0, p_l))

                evaluate('XGBoost', xgb_p)
                evaluate('LightGBM', lgb_p)

        elif segment == 'B — Mid':
            # 1. MA kết hợp Trend
            evaluate('MA + Trend', ma_trend_forecast(train_df['qty'].values, 3))

            # 2. Linear Regression (ML Nhẹ)
            df_train_clean = prepare_ml_data(train_df)
            if len(df_train_clean) > 0:
                X_train, y_train = df_train_clean[['Lag_1', 'Lag_2', 'Lag_3']], df_train_clean['qty']
                m_lr = LinearRegression().fit(X_train, y_train)
                lr_p, last_lr = [], train_df['qty'].tolist()
                for i in range(3):
                    p_l = m_lr.predict(pd.DataFrame([[last_lr[-1], last_lr[-2], last_lr[-3]]], columns=['Lag_1', 'Lag_2', 'Lag_3']))[0]
                    lr_p.append(max(0, p_l)); last_lr.append(max(0, p_l))
                evaluate('Linear Regression', lr_p)

        elif segment == 'D — Dormant':
            t_qty = train_df['qty'].values

            # 1. Croston
            evaluate('Croston', croston_forecast(t_qty, 3))

            # 2. Decay Forecast (Giảm nhanh 40% mỗi tháng)
            last_actual = t_qty[-1]
            decay_p = [last_actual * (0.6 ** 1), last_actual * (0.6 ** 2), last_actual * (0.6 ** 3)]
            evaluate('Decay Forecast', decay_p)

        # --------- BƯỚC 5: TẠO CỘT GHI MÔ HÌNH TỐI ƯU ---------
        if not mapes:
            continue # Bỏ qua nếu không có đủ dữ liệu để tính toán

        best_model = min(mapes, key=mapes.get)

        row_data = {
            'Phân khúc': segment,
            'Mã Hàng (F_SKU)': fsku,
            'Các thuật toán thi đấu': ' vs '.join(list(mapes.keys())),
            '🏆 MÔ HÌNH TỐI ƯU': best_model,
            'MAPE Tối ưu (%)': round(mapes[best_model], 2),
            'RMSE Tối ưu': round(rmses[best_model], 2)
        }

        # Ghi chú chi tiết kết quả của từng thuật toán để dễ so sánh trong Excel
        for m in ['XGBoost', 'LightGBM', 'Linear Regression', 'MA + Trend', 'Croston', 'Decay Forecast']:
            row_data[f'{m} (MAPE %)'] = round(mapes.get(m, np.nan), 2)
            row_data[f'{m} (RMSE)'] = round(rmses.get(m, np.nan), 2)

        results.append(row_data)


# =====================================================================
# BƯỚC 6: XUẤT FILE CSV KẾT QUẢ
# =====================================================================
df_report = pd.DataFrame(results)

# Sắp xếp theo Phân khúc và FSKU để dễ đọc
df_report = df_report.sort_values(by=['Phân khúc', 'Mã Hàng (F_SKU)'])

# Xuất ra CSV
OUTPUT_FILE = 'Ket_Qua_Thi_Dau_FSKU_Nhom_A_B_D.csv'
df_report.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print("\n" + "="*70)
print("✅ HOÀN TẤT TOÀN BỘ QUY TRÌNH!")
print(f"👉 File kết quả cuối cùng: {OUTPUT_FILE}")
print("="*70)

BẮT ĐẦU KHỞI CHẠY HỆ THỐNG ĐÁNH GIÁ (NHÓM A, B, D)

⚙️ Đang thi đấu tại Phân khúc: [A — Core ML] (75 FSKU)

⚙️ Đang thi đấu tại Phân khúc: [D — Dormant] (9 FSKU)

⚙️ Đang thi đấu tại Phân khúc: [B — Mid] (4 FSKU)

✅ HOÀN TẤT TOÀN BỘ QUY TRÌNH!
👉 File kết quả cuối cùng: Ket_Qua_Thi_Dau_FSKU_Nhom_A_B_D.csv
